# 🛡️ Quality Control in Climatology Engine

This notebook introduces the Quality Flag system that evaluates the quality of distribution fits.

**What you will learn:**
- The concept of Quality Flag in distribution fitting
- Types of quality flags (PASS, LOW_SAMPLE, NO_CONVERGENCE, OUTLIER, HIGH_AICC, BAD_SKEW)
- How to evaluate fit quality
- Setting thresholds for each flag
- Displaying fit quality for different models
- Interpreting Quality Flag results

---

## 📐 Introduction to Quality Flag System

The Quality Flag system automatically evaluates each fit and assigns appropriate flags.

### Quality Flags

| Code | Name | Description |
|------|------|-------------|
| 0 | PASS | Successful fit |
| 1 | LOW_SAMPLE | Sample size below minimum (3) |
| 2 | NO_CONVERGENCE | Optimization did not converge |
| 3 | OUTLIER | Outliers detected in data |
| 4 | HIGH_AICC | AICc exceeds threshold |
| 5 | BAD_SKEW | Skewness outside acceptable range |
| 6 | NAN_INPUT | Data contains NaN values |
| 7 | INF_INPUT | Data contains Inf values |

### Default Thresholds

| Parameter | Default Value | Description |
|-----------|---------------|-------------|
| `min_sample_size` | 3 | Minimum sample size for fitting |
| `threshold_aicc` | 1000 | Maximum allowed AICc |
| `threshold_skew` | 5.0 | Maximum absolute skewness allowed |
| `outlier_sigma` | 4.0 | Sigma threshold for outlier detection |

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from core.engine.plugin_loader import load_plugins
from core.quality.quality_flag import QualityFlag

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11
print('✅ Libraries loaded.')

In [ ]:
# Display quality flags
flag_names = {
    QualityFlag.PASS: '✅ PASS',
    QualityFlag.LOW_SAMPLE: '⚠️ LOW_SAMPLE',
    QualityFlag.NO_CONVERGENCE: '❌ NO_CONVERGENCE',
    QualityFlag.OUTLIER: '⚠️ OUTLIER',
    QualityFlag.HIGH_AICC: '⚠️ HIGH_AICC',
    QualityFlag.BAD_SKEW: '⚠️ BAD_SKEW',
    QualityFlag.NAN_INPUT: '❌ NAN_INPUT',
    QualityFlag.INF_INPUT: '❌ INF_INPUT',
}

print("📋 Quality Flags:")
print("=" * 50)
for code, name in flag_names.items():
    print(f"   {code}: {name}")
print("=" * 50)

In [ ]:
# Load distribution plugins
plugins = load_plugins()
print(f'✅ Number of loaded distributions: {len(plugins)}')

for code, dist in plugins.items():
    print(f"   [{code}] {dist.name} (params: {dist.params})")

distributions = {dist.name: dist for dist in plugins.values()}

In [ ]:
# Load sample data
sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))
data = station_data.values

# Select tmean data for one year
data_year = data[:365, 1]

print(f'📊 Number of samples: {len(data_year)}')
print(f'   Mean: {np.mean(data_year):.2f}°C')
print(f'   Standard deviation: {np.std(data_year):.2f}°C')

In [ ]:
# Fit all distributions and evaluate quality
quality_results = []
print("\n🔄 Fitting and evaluating quality...\n")

for name, dist in distributions.items():
    try:
        res = dist.fit(data_year)
        flags = QualityFlag.evaluate(res, data_year)
        quality_results.append({
            'Distribution': name,
            'AICc': res.get('aicc', np.nan),
            'Flags': flags,
            'Flag_Count': len(flags),
            'Status': 'PASS' if QualityFlag.PASS in flags else 'FAIL'
        })
        flag_str = ', '.join([flag_names.get(f, str(f)) for f in flags])
        print(f"{'✅' if QualityFlag.PASS in flags else '⚠️'} {name}: {flag_str}")
    except Exception as e:
        print(f"❌ {name}: Error - {str(e)}")

print("\n✅ Quality evaluation complete.")

In [ ]:
# Create quality table
quality_df = pd.DataFrame(quality_results)
quality_df['Flag_Count'] = quality_df['Flags'].apply(len)
quality_df = quality_df.sort_values('AICc').reset_index(drop=True)
quality_df.index = quality_df.index + 1

# Display table
print("📊 Model Quality Table:")
print("=" * 80)
display_cols = ['Distribution', 'AICc', 'Flag_Count', 'Status']
quality_df[display_cols]

In [ ]:
# Display detailed flags for each model
print("📋 Detailed Quality Flags:")
print("=" * 80)
for _, row in quality_df.iterrows():
    flags = row['Flags']
    flag_str = ', '.join([flag_names.get(f, str(f)) for f in flags])
    print(f"\n{row['Distribution']} ({row['Status']}):")
    print(f"   AICc: {row['AICc']:.2f}")
    print(f"   Flags: {flag_str if flags else 'None'}")

In [ ]:
# Function to count flag distribution
def get_flag_counts(quality_df):
    """Calculate count of each flag type"""
    all_flags = []
    for flags in quality_df['Flags']:
        all_flags.extend(flags)
    counts = {}
    for f in all_flags:
        counts[f] = counts.get(f, 0) + 1
    return counts

flag_counts = get_flag_counts(quality_df)

print("📊 Quality Flag Distribution:")
print("=" * 50)
for code, count in sorted(flag_counts.items()):
    name = flag_names.get(code, f'UNKNOWN_{code}')
    print(f"   {name}: {count}")
print("=" * 50)

In [ ]:
# Plot flag distribution
if flag_counts:
    fig, ax = plt.subplots(figsize=(10, 6))

    labels = [flag_names.get(code, f'UNKNOWN_{code}') for code in flag_counts.keys()]
    values = list(flag_counts.values())

    colors = ['#2ecc71' if code == 0 else '#e74c3c' if code in [1, 2, 6, 7] else '#f39c12' 
              for code in flag_counts.keys()]

    bars = ax.bar(labels, values, color=colors, alpha=0.7, edgecolor='black', linewidth=1)

    ax.set_xlabel('Flag Type', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Quality Flag Distribution Across Models', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

    plt.tight_layout()
    plt.show()

In [ ]:
# Analyze PASS and FAIL models
pass_models = quality_df[quality_df['Status'] == 'PASS']
fail_models = quality_df[quality_df['Status'] == 'FAIL']

print(f"✅ PASS models: {len(pass_models)}")
if len(pass_models) > 0:
    print(f"   {', '.join(pass_models['Distribution'].tolist())}")

print(f"\n❌ FAIL models: {len(fail_models)}")
if len(fail_models) > 0:
    print(f"   {', '.join(fail_models['Distribution'].tolist())}")

In [ ]:
# Re-evaluate with custom thresholds
print("\n🔧 Re-evaluating with custom thresholds:")
print("   - min_sample_size: 5")
print("   - threshold_aicc: 500")
print("   - threshold_skew: 3.0")
print("   - outlier_sigma: 3.0")

quality_results_new = []
for name, dist in distributions.items():
    try:
        res = dist.fit(data_year)
        flags = QualityFlag.evaluate(res, data_year, threshold_aicc=500)
        quality_results_new.append({
            'Distribution': name,
            'AICc': res.get('aicc', np.nan),
            'Flags': flags,
            'Status': 'PASS' if QualityFlag.PASS in flags else 'FAIL'
        })
    except:
        pass

quality_df_new = pd.DataFrame(quality_results_new)

# Compare results before and after
pass_old = len(quality_df[quality_df['Status'] == 'PASS'])
pass_new = len(quality_df_new[quality_df_new['Status'] == 'PASS'])

print(f"\n📊 Comparison:")
print(f"   Before threshold change: {pass_old} PASS models")
print(f"   After threshold change: {pass_new} PASS models")

## 📋 Summary

In this notebook you learned:

✅ Quality Flag system and flag types
✅ How to evaluate fit quality with `QualityFlag.evaluate()`
✅ Displaying quality flags for each model
✅ Analyzing flag distribution across models
✅ Impact of threshold settings on quality results

---

**Key Takeaways:**

1. `PASS` flag indicates a successful fit.
2. `LOW_SAMPLE` and `NAN_INPUT` flags indicate data issues.
3. `HIGH_AICC` flag indicates low model quality.
4. Thresholds can be adjusted based on project requirements.
5. Models with `PASS` flag are more reliable.

---

**Next Steps:**
- Notebook 06: Bootstrap Uncertainty
- Notebook 07: Parallel Processing
- Notebook 08: Visualization